This notebook generates n simulated replicates of the shendure dataset. As though the whole experiment were run n more times. 

Imports

In [1]:
import scMPRAforge as scm
from dask.distributed import Client, LocalCluster

2025-10-28 14:53:37.873383: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-10-28 14:53:37.877569: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /vast/palmer/apps/avx2/software/code-server/4.91.1/lib:/vast/palmer/apps/avx2/software/gettext/0.22.5-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/libiconv/1.17-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/ncurses/6.5-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/libxml2/2.12.7-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/XZ/5.4.5-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/expat/2.6.2-GCCcore-13.3.0/lib:/vast/palmer/apps/av

In [2]:
%load_ext autoreload
%autoreload 2

Set up cluster

In [3]:
cluster=LocalCluster(memory_limit='8GB')
client=Client(cluster)

Load shendure ortho

In [4]:
path="/gpfs/gibbs/pi/reilly/tabula_data/shendure"
name="ortho_primordial_v3"
data_root="/gpfs/gibbs/pi/reilly/tabula_data"
primordial=scm.ortho.load(client,path,name)

Create a ground-truth dataframe from the description.

In [5]:
description=scm.describe_parameters(primordial.by_cell_type_parameters,
                        dat=primordial.training_data.data,
                        split="cell_type")
gt=description[["cell_type","cre_id","mu"]].groupby(["cell_type","cre_id"]).agg(true_mean=("mu","mean")).reset_index()
gt=scm.zero_pad_ground_truth(gt)
gt

,cell_type,cre_id,true_mean
0,Cardiomyocytes,Bend5_chr4_8174,0.064510
1,Cardiomyocytes,Bend5_chr4_8175,1.636992
2,Cardiomyocytes,Bend5_chr4_8179,0.053898
3,Cardiomyocytes,Btg1_chr10_9578,3.505954
4,Cardiomyocytes,Cdk5r1_chr11_12559,0.283988
...,...,...,...
1856,SurfaceEctoderm,Tubb2b_chr13_2578,0.000000
1863,SurfaceEctoderm,Txndc12_chr4_7969,0.000000
1864,SurfaceEctoderm,Txndc12_chr4_7971,0.000000
1866,SurfaceEctoderm,Txndc12_chr4_7975,0.000000


Abstract what the library looked like from the ortho training data. We will make the simplifying assumption that all MPRA bc are equal abundance in the original & none were lost.

In [6]:
library=primordial.training_data.data[["cre_id","mpra_bc"]].drop_duplicates()
abundance = 1/len(library)
library["abundance"]=abundance
library

,cre_id,mpra_bc,abundance
0,Txndc12_chr4_7978,ACGTAACATTATAAT,0.000035
1,Klf4_chr4_3952,TGTTTAAGTCAACAA,0.000035
2,Foxa2_chr2_13840,CAACAACACATTTTA,0.000035
3,reference,TACCTAATGGGAAAG,0.000035
4,Lamc1_chr1_12152,AAATGGTAAAAGGCA,0.000035
...,...,...,...
751918,Epas1_chr17_10116,GGAACTCAGTCCGTT,0.000035
751919,Bend5_chr4_8174,AATTACAACTACCAA,0.000035
758757,Sparc_chr11_7233,GAGCGGGTTCAAGAT,0.000035
777357,Foxa2_chr2_13830,TACAATGCCCATTAT,0.000035


Create object

In [7]:
simu_obj=scm.de_novo_simulation(
                        simulation_replicates=5,
                        experiment_bounds=scm.SHENDURE_BOUNDS,
                        ground_truth=gt,
                        library=library)

Simulate

In [8]:
simu_obj.gamut(client)

scMPRAforge: INFO: 2290/256574 cells (0.893%) have ≥1 multi-transfection event.


In [9]:
DATA_ROOT="/gpfs/gibbs/pi/reilly/tabula_data"

In [10]:
simu_obj.save(path=f"{DATA_ROOT}/simulated",name="shendure_calibrated_sim_20251028_erin")

scMPRAforge: INFO: 2278/256498 cells (0.888%) have ≥1 multi-transfection event.
scMPRAforge: INFO: 2346/256697 cells (0.914%) have ≥1 multi-transfection event.
scMPRAforge: INFO: 2355/256623 cells (0.918%) have ≥1 multi-transfection event.
scMPRAforge: INFO: 2281/256604 cells (0.889%) have ≥1 multi-transfection event.


In [12]:
cluster.close()
client.close()